# The raw model's critic has nothing to read

The first notebook of a series on **Qwen3.5-0.8B**, and the question it was set: reproduce the
configuration the 1.7B series settled on for the **v2 prompt, unchanged** -- actor-critic PPO whose
reward is paid **per field, at the token that writes the field**, not once at the end -- with the
calculator tool and the SFT seed taken away, on the raw model. Tune the critic first; then train.

**The critic cannot be tuned, because there is nothing for it to learn.** Measured three ways:

| | what it measures | result |
| --- | --- | --- |
| online probe | the in-loop critic protocol replayed on 320 frozen rollouts, 8 layers x 3 rates | best **+0.016** over a position-only clock, ahead on 53% of batches |
| ridge, held out by prompt | the best any linear head on the policy's state could do | position alone **+0.023**; every hidden layer makes it **worse** (-0.008 to -0.014) |
| a training run | the same critic while the policy learns | its own pre-set gate stopped it at step 75: **-0.018** over the clock, while the diagnosis collapsed to one label on all 200 cases |

What the return depends on is the answer key: with the true `root_cause` added, the same ridge gains
**+0.058**. The raw model gets the numbers right 1% of the time and the cause at chance, so its state
does not know whether what it is writing is right -- and a value function has to. Every run of the
1.7B series whose critic got ahead of the clock had the numbers right first; `02` gives this model
a calculator.

In [1]:
import importlib.util, json, random, statistics as st, sys
from collections import deque
from pathlib import Path

import torch

HERE = Path.cwd() if (Path.cwd() / "01_raw_dense_critic.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's own module. The file name starts with a digit, so it is loaded by path.
spec = importlib.util.spec_from_file_location("raw01", HERE / "01_raw_dense_critic.py")
raw01 = importlib.util.module_from_spec(spec)
sys.modules["raw01"] = raw01
spec.loader.exec_module(raw01)
PREFIX = raw01.PREFIX
GPU = torch.cuda.is_available()
print(raw01.MODEL, "| prompt", raw01.PROMPT_VERSION, "| LoRA on", ", ".join(raw01.LORA_TARGETS))

Qwen/Qwen3.5-0.8B | prompt v2 | LoRA on q_proj, k_proj, v_proj, o_proj, in_proj_qkv, in_proj_z, in_proj_a, in_proj_b, out_proj


## The brief, and the model

**Task and scoring are imported, never copied**: `task.prompt.build_messages(record, "v2")`, the
`reward.score` grader under the `ABLATE` weights, `eval.generate_hf` and `eval.summarise` for every
held-out number -- the same functions every run in this repository is measured with.

**The model.** `Qwen/Qwen3.5-0.8B` has 24 layers, three of every four a GatedDeltaNet
*linear-attention* layer and every fourth full attention; hidden size 1024; vocabulary 248,320.
LoRA (r = 16) covers the mixing projections of both kinds. Thinking is switched off through the chat
template, as `eval.py` does.

**The loop** (`01_raw_dense_critic.py`, this notebook's own file): one sample per training prompt,
8 prompts per step, temperature 1.0; each field's credit placed on the token that completes it;
a linear value head on one detached hidden layer; GAE with gamma = 1, lambda = 0.95; the head fitted
for 8 Adam steps over the last 25 batches; 4 clipped-surrogate epochs on the actor. Held out every
25 steps on the 200 dev cases, greedy.

**The gates, fixed before any run** -- a run stops itself on any of them:

* *no regression*: held-out reward more than 0.05 below step 0's, twice in a row;
* *critic*: from step 40, at every evaluation, the median of (critic's explained variance - the
  clock's) over the last 25 steps must be above 0; two failures in a row stop the run;
* *convergence*: eight evaluations without a new best held-out reward.

## The prompt is v2's

The rendered prompt contains v2's system and user turns verbatim. Everything the chat template adds
around them is printed below: role markers and the empty think block that `enable_thinking=False`
writes.

In [2]:
from transformers import AutoTokenizer
from task.prompt import build_messages, build_user_prompt, SYSTEM_PROMPTS

tok = AutoTokenizer.from_pretrained(raw01.MODEL, padding_side="left")
case = raw01.load_cases("dev")[0]
text = raw01.render_prompt(tok, case["record"])
system, user = SYSTEM_PROMPTS["v2"], build_user_prompt(case["record"], "v2")
assert system in text and user in text
rest = text.replace(system, "<v2 system>").replace(user, "<v2 user>")
print("what the chat template adds around v2's two messages, verbatim:\n")
print(rest)

/home/bayan/MembraneClaw/experiments/membrane_grpo/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


what the chat template adds around v2's two messages, verbatim:

<|im_start|>system
<v2 system><|im_end|>
<|im_start|>user
<v2 user><|im_end|>
<|im_start|>assistant
<think>

</think>




## Left padding through the linear-attention layers

One sample from each of eight prompts makes a left-padded batch. For full attention the mask and
the position ids handle that; a linear-attention layer is a recurrence, and padding that leaked into
its state would corrupt every log-probability without raising an error. The check scores the same
sampled completions padded and alone, in bf16 and in float32, against float32 unpadded, on the
tokens that are actually scored.

**Padding is exact to float32 precision**: 0.0002 nats per token, against 0.011 that bf16 rounding
costs whether a row is padded or not. The layers zero padded positions before the convolution and
the recurrence, so the state never sees them.

In [3]:
CHECKS = HERE / f"{PREFIX}_checks.json"

def padding_check(n=8, max_new=300):
    # Score the same sampled completions two ways -- inside a left-padded batch, and
    # alone with no padding -- in bf16 and in float32, against float32 unpadded.
    from transformers import AutoModelForCausalLM
    from grpo_scratch import build_mask, selective_logprobs
    cases = raw01.load_cases("train")[:n]
    texts = [raw01.render_prompt(tok, c["record"]) for c in cases]
    models = {d: AutoModelForCausalLM.from_pretrained(raw01.MODEL, dtype=d).to("cuda").eval()
              for d in (torch.bfloat16, torch.float32)}
    enc = tok(texts, return_tensors="pt", padding=True).to("cuda")
    P = enc["input_ids"].shape[1]
    torch.manual_seed(0)
    with torch.no_grad():
        out = models[torch.bfloat16].generate(**enc, do_sample=True, temperature=1.0, top_p=1.0,
                                              max_new_tokens=max_new, pad_token_id=tok.pad_token_id)
    comp = out[:, P:]
    L = comp.shape[1]
    mask = build_mask(comp, tok.eos_token_id, tok.pad_token_id)
    attn = torch.cat([enc["attention_mask"], mask.long()], 1)

    def lp(model, seq, a=None):
        with torch.no_grad():
            o = model(input_ids=seq, attention_mask=a,
                      position_ids=None if a is None else raw01.padded_position_ids(a), logits_to_keep=L + 1)
        return selective_logprobs(o.logits[:, :-1], seq[:, -L:])

    rows = []
    for i in range(n):
        real = enc["attention_mask"][i].bool()
        alone = torch.cat([enc["input_ids"][i][real], comp[i]]).unsqueeze(0)
        m = mask[i].bool()
        ref = lp(models[torch.float32], alone)[0]
        diff = lambda x: round(float((x - ref)[m].abs().mean()), 5)
        rows.append({
            "pads": int((~real).sum()), "tokens": int(m.sum()),
            "f32_padded": diff(lp(models[torch.float32], out[i:i + 1], attn[i:i + 1])[0]),
            "bf16_unpadded": diff(lp(models[torch.bfloat16], alone)[0]),
            "bf16_padded": diff(lp(models[torch.bfloat16], out[i:i + 1], attn[i:i + 1])[0]),
        })
    del models
    torch.cuda.empty_cache()
    return rows

if not CHECKS.exists():
    CHECKS.write_text(json.dumps({"padding": padding_check()}, indent=2) + "\n")
checks = json.loads(CHECKS.read_text())

print("mean |log p| difference per scored token, against float32 with no padding\n")
print(f"  {'row':>3} {'pads':>5} {'tokens':>7} | {'f32 padded':>11} | {'bf16 unpadded':>14} {'bf16 padded':>12}")
for i, r in enumerate(checks["padding"]):
    print(f"  {i:>3} {r['pads']:>5} {r['tokens']:>7} | {r['f32_padded']:>11.5f} | "
          f"{r['bf16_unpadded']:>14.5f} {r['bf16_padded']:>12.5f}")
mean = lambda k: st.mean(r[k] for r in checks["padding"])
print(f"  {'mean':>17} | {mean('f32_padded'):>11.5f} | {mean('bf16_unpadded'):>14.5f} {mean('bf16_padded'):>12.5f}")

mean |log p| difference per scored token, against float32 with no padding

  row  pads  tokens |  f32 padded |  bf16 unpadded  bf16 padded
    0    10      80 |     0.00035 |        0.01094      0.01097
    1     0     300 |     0.00000 |        0.01399      0.01399
    2     8       9 |     0.00064 |        0.03233      0.02115
    3     7     102 |     0.00017 |        0.00536      0.00494
    4     1      85 |     0.00011 |        0.00505      0.00495
    5     5     102 |     0.00008 |        0.00237      0.00156
    6     2      87 |     0.00009 |        0.00981      0.00793
    7     2      84 |     0.00021 |        0.00812      0.00709
               mean |     0.00021 |        0.01100      0.00907


## Tuning the critic: what the online protocol can reach

320 completions of the frozen model, sampled exactly as a training step samples them, with the
per-field credit the actor is paid. The critic's protocol is replayed on them: score the head on a
batch *before* fitting on it, then 8 Adam steps over the last 25 batches. Eight layers x three step
sizes; the learning rate of each is set so that one Adam step moves V by the same amount,
`|dV| ~ lr * ||h||_1`, because the layers' feature norms differ by a factor of 90.

The **clock** is the yardstick: the mean return at each relative position of the answer, fitted on
the batch itself. With per-field credit the return falls as the answer is written, so a head that
knew only how far along it was would already explain part of it; a critic is worth having only by
what it explains above that. The clock is fitted in-sample and the critic is not, so beating it is
the conservative test.

**No configuration beats it by more than noise.** The best is +0.016, ahead on 53% of batches.

In [4]:
PROBE_CACHE = HERE / f"{PREFIX}_probe_cache.pt"
PROBE = HERE / f"{PREFIX}_probe.json"
LAYERS = (24, 23, 22, 20, 18, 16, 12, 8)   # indices into hidden_states; 24 is the final, normed one
N_BATCHES, BATCH = 40, 8

def collect_probe():
    # Rollouts of the frozen policy, sampled exactly as a training step samples them:
    # 8 train prompts, temperature 1.0, the dense credit the actor is paid (flat x3).
    # A fresh LoRA has B = 0, so the policy here *is* the raw model.
    torch.manual_seed(1)
    rng = random.Random(1)
    policy, ptok = raw01.load_policy(raw01.MODEL, 16, "cuda")
    policy.eval()
    cases = raw01.load_cases("train")
    batches = []
    for k in range(N_BATCHES):
        chunk = [rng.choice(cases) for _ in range(BATCH)]
        roll = raw01.rollout(policy, ptok, chunk, max_new_tokens=640, temperature=1.0, weights=raw01.ABLATE)
        L = roll.mask.shape[1]
        ids = roll.sequences[:, -L:]
        paid = raw01.dense_rewards(roll.completions, ids, roll.mask, roll.cases, raw01.ABLATE, ptok, 3.0)
        plain = raw01.dense_rewards(roll.completions, ids, roll.mask, roll.cases, raw01.ABLATE, ptok, 1.0)
        with torch.no_grad():
            hs = []
            for i in range(BATCH):
                o = policy(input_ids=roll.sequences[i:i + 1], attention_mask=roll.attention_mask[i:i + 1],
                           position_ids=raw01.padded_position_ids(roll.attention_mask[i:i + 1]),
                           logits_to_keep=1, output_hidden_states=True)
                hs.append(torch.stack([o.hidden_states[l][0, -(L + 1):-1] for l in LAYERS]))
            hs = torch.stack(hs, 1)                      # (layers, batch, L, hidden)
        sel = roll.mask > 0
        # gamma = lam = 1: the Monte-Carlo return, which does not depend on any V.
        returns = paid.flip(-1).cumsum(-1).flip(-1) * roll.mask
        batches.append({
            "hidden": hs[:, sel].cpu(), "returns": returns.cpu(), "mask": roll.mask.cpu(),
            "rewards": roll.rewards, "parsed": roll.parsed, "truncated": roll.truncated,
            "numeric": raw01.sampled_numeric(roll), "completions": roll.completions,
            "sum_err": max(abs(float(plain[b].sum()) - roll.rewards[b]) for b in range(BATCH)),
        })
        print(f"  batch {k + 1}/{N_BATCHES}  L {L}", end="\r")
    del policy
    torch.cuda.empty_cache()
    torch.save(batches, PROBE_CACHE)
    return batches

batches = torch.load(PROBE_CACHE, weights_only=False) if PROBE_CACHE.exists() else collect_probe()
seqs = N_BATCHES * BATCH
rewards = [r for b in batches for r in b["rewards"]]
print(f"frozen policy, {seqs} sampled completions at temperature 1.0 on train")
print(f"  reward mean {st.mean(rewards):.4f}   parsed {sum(p for b in batches for p in b['parsed']) / seqs:.3f}"
      f"   ran to 640 tokens {sum(t for b in batches for t in b['truncated']) / seqs:.3f}"
      f"   numeric {st.mean(b['numeric'] for b in batches):.3f}")
print(f"  dense credit at flat x1 sums to reward.score: max |row sum - score| = "
      f"{max(b['sum_err'] for b in batches):.2e}")

frozen policy, 320 sampled completions at temperature 1.0 on train
  reward mean 0.1772   parsed 0.872   ran to 640 tokens 0.013   numeric 0.012
  dense credit at flat x1 sums to reward.score: max |row sum - score| = 4.53e-08


In [5]:
def fit_online(li, lr, bias, epochs=8, window=25, clip=0.2):
    # The in-loop critic protocol, replayed on the cached batches: score the head on a
    # batch *before* it has seen it, then fit 8 Adam steps over the last 25 batches.
    head = raw01.ValueHead(batches[0]["hidden"].shape[-1], bias).cuda()
    opt = torch.optim.AdamW(head.parameters(), lr=lr, weight_decay=0.0)
    hist, rows = deque(maxlen=window), []
    for b in batches:
        h, mask, G = b["hidden"][li].cuda(), b["mask"].cuda(), b["returns"].cuda()
        with torch.no_grad():
            v = head(h)
        V = torch.zeros_like(G)
        V[mask > 0] = v
        rows.append(raw01.critic_fit(V, G, mask))
        hist.append((h, v, G[mask > 0]))
        H, OV, R = (torch.cat([x[j] for x in hist]) for j in range(3))
        for _ in range(epochs):
            opt.zero_grad(set_to_none=True)
            loss, _ = raw01.value_loss(head(H), OV, R, clip_eps=clip)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(head.parameters(), 1.0)
            opt.step()
    return rows

def summary(rows, skip=10):
    rows = [r for r in rows[skip:] if r["value_ev"] is not None]
    res = [r["value_ev"] - r["value_ev_position"] for r in rows]
    return {"ev": st.median(r["value_ev"] for r in rows), "clock": st.median(r["value_ev_position"] for r in rows),
            "residual": st.median(res), "ahead": sum(x > 0 for x in res) / len(res)}

# The starting value: the constant that minimises the head's squared error before it
# has learned anything -- the mean return over every scored token.
tokens_G = torch.cat([b["returns"][b["mask"] > 0] for b in batches])
BIAS = round(float(tokens_G.mean()), 4)
# The learning rate, per layer. Adam moves each of the head's 1024 weights by about lr
# per step, and those steps add up through the features: |dV| ~ lr * ||h||_1.
norm1 = {l: float(torch.cat([b["hidden"][i] for b in batches]).float().abs().sum(-1).median())
         for i, l in enumerate(LAYERS)}
DV = (0.02, 0.07, 0.2)

if PROBE.exists():
    probe = json.loads(PROBE.read_text())
    probe["norm1"] = {int(k): v for k, v in probe["norm1"].items()}
else:
    probe = {"bias": BIAS, "norm1": norm1, "configs": []}
    for i, l in enumerate(LAYERS):
        for dv in DV:
            lr = dv / norm1[l]
            probe["configs"].append({"layer": l, "dv": dv, "lr": lr, **summary(fit_online(i, lr, BIAS))})
    PROBE.write_text(json.dumps(probe, indent=2) + "\n")

print(f"starting value (mean return over scored tokens): {probe['bias']:.4f}   "
      f"mean sequence reward: {st.mean(rewards):.4f}\n")
print("held out by batch, batches 11-40: median explained variance of the return")
print(f"  {'layer':>5} {'||h||_1':>8} {'|dV|':>5} {'lr':>9} | {'critic':>7} {'clock':>7} {'residual':>9} {'ahead':>6}")
for c in probe["configs"]:
    print(f"  {c['layer']:>5} {probe['norm1'][c['layer']]:>8.0f} "
          f"{c['dv']:>5} {c['lr']:>9.2e} | {c['ev']:>+7.3f} {c['clock']:>+7.3f} {c['residual']:>+9.4f} {c['ahead']:>6.0%}")

starting value (mean return over scored tokens): 0.1583   mean sequence reward: 0.1772

held out by batch, batches 11-40: median explained variance of the return
  layer  ||h||_1  |dV|        lr |  critic   clock  residual  ahead
     24     3169  0.02  6.31e-06 |  +0.097  +0.079   +0.0121    53%
     24     3169  0.07  2.21e-05 |  +0.081  +0.079   -0.0078    47%
     24     3169   0.2  6.31e-05 |  +0.077  +0.079   -0.0178    40%
     23      209  0.02  9.55e-05 |  +0.116  +0.079   +0.0157    53%
     23      209  0.07  3.34e-04 |  +0.097  +0.079   -0.0011    50%
     23      209   0.2  9.55e-04 |  +0.101  +0.079   +0.0060    50%
     22      186  0.02  1.08e-04 |  +0.099  +0.079   +0.0117    53%
     22      186  0.07  3.77e-04 |  +0.090  +0.079   -0.0068    50%
     22      186   0.2  1.08e-03 |  +0.106  +0.079   -0.0097    47%
     20      145  0.02  1.38e-04 |  +0.095  +0.079   +0.0030    50%
     20      145  0.07  4.82e-04 |  +0.091  +0.079   -0.0187    43%
     20      145   0.2

## Is there anything to read? A ridge upper bound

The online probe learns one batch at a time. A ridge regression fitted on all 320 rollouts at once,
**held out by prompt** (five folds over 231 prompts) and given relative position as well, is the
best any linear head could do with these features.

Position alone explains **+0.023** of the return's variance held out -- the clock's +0.079 above is
fitted in-sample. **Adding any hidden layer makes it worse**, and the ridge picks its strongest
penalty every time: the policy's state carries nothing about the return that position does not.

The last row gives the head something the policy never sees, the case's true `root_cause`
(interacted with five position bins): **+0.058** over position. The return depends on whether the
answer is right, and whether it is right is not in the state of a model that gets the cause right at
chance.

In [6]:
from task.decision_table import CAUSES

rng = random.Random(1)            # the batches collect_probe drew, prompt by prompt
train_cases = raw01.load_cases("train")
drawn = [[rng.choice(train_cases) for _ in range(BATCH)] for _ in range(N_BATCHES)]
device = "cuda" if torch.cuda.is_available() else "cpu"

G, POS, CID, CAUSE = [], [], [], []
for b, chunk in zip(batches, drawn):
    m = b["mask"] > 0
    rel = (torch.arange(m.shape[1]).unsqueeze(0) / m.sum(-1, keepdim=True).clamp(min=1) * 20).floor().clamp(max=19).long()
    G.append(b["returns"][m]); POS.append(rel[m])
    for i, case in enumerate(chunk):
        k = int(m[i].sum())
        CID += [case["id"]] * k
        CAUSE += [[float(case["answer"]["root_cause"] == c) for c in CAUSES]] * k
G, POS = torch.cat(G).to(device), torch.cat(POS).to(device)
ids = sorted(set(CID))
fold = torch.tensor([ids.index(c) for c in CID]) % 5
P = torch.nn.functional.one_hot(POS, 20).float()

def ridge_cv(X, lam):
    pred = torch.zeros_like(G)
    for f in range(5):
        tr, te = (fold != f).to(device), (fold == f).to(device)
        mu, sd = X[tr].mean(0), X[tr].std(0).clamp(min=1e-6)
        Z, ym = (X[tr] - mu) / sd, G[tr].mean()
        w = torch.linalg.solve(Z.T @ Z + lam * torch.eye(Z.shape[1], device=device), Z.T @ (G[tr] - ym))
        pred[te] = ((X[te] - mu) / sd) @ w + ym
    return 1 - float(((G - pred) ** 2).mean() / G.var(unbiased=False))

best = lambda X: max(ridge_cv(X, lam) for lam in (1e-1, 1, 1e1, 1e2, 1e3, 1e4, 1e5))
base = best(P)
print(f"held out by prompt ({len(ids)} prompts, {len(G)} scored tokens)\n")
print(f"  position (20 bins)                      {base:+.4f}")
for i, l in enumerate(LAYERS):
    X = torch.cat([b["hidden"][i].float() for b in batches]).to(device)
    ev = best(torch.cat([P, X], 1))
    print(f"  position + hidden_states[{l:>2}]          {ev:+.4f}   over position {ev - base:+.4f}")
    del X
C = torch.tensor(CAUSE, device=device)
bins5 = torch.nn.functional.one_hot(POS // 4, 5).float()
priv = torch.cat([C * bins5[:, j:j + 1] for j in range(5)], 1)
ev = best(torch.cat([P, priv], 1))
print(f"  position + the true root_cause          {ev:+.4f}   over position {ev - base:+.4f}   (not in the policy's state)")

held out by prompt (231 prompts, 37988 scored tokens)

  position (20 bins)                      +0.0234


  position + hidden_states[24]          +0.0150   over position -0.0084


  position + hidden_states[23]          +0.0141   over position -0.0093


  position + hidden_states[22]          +0.0127   over position -0.0107


  position + hidden_states[20]          +0.0117   over position -0.0117


  position + hidden_states[18]          +0.0099   over position -0.0135


  position + hidden_states[16]          +0.0103   over position -0.0130


  position + hidden_states[12]          +0.0151   over position -0.0083


  position + hidden_states[ 8]          +0.0126   over position -0.0107
  position + the true root_cause          +0.0817   over position +0.0584   (not in the policy's state)


## Training says the same

One run of the brief as written, with the critic the probe chose (`hidden_states[23]`,
`lr = 0.02 / ||h||_1`, bias 0.1583, the mean return over scored tokens) -- `ent0-s0`, launched by
`01_raw_dense_critic_run_ent0.sh`. Its entropy bonus is 0: with the series' default of 0.005 the raw
model's answers ran away from the JSON to the 640-token budget (104 -> 591 tokens within 40 steps).

It **stopped itself at step 75 on the critic gate**. From step 25 the critic sits on the clock
(-0.0003 over steps 15-39, 12 of 25 ahead) and then falls behind it (-0.016 over steps 40-74, 6 of 35
ahead) -- the window in which every successful 1.7B run pulled ahead (below). Meanwhile the policy
did learn one thing, the flags (0.360 -> 0.512) and the schema (0.925 -> 1.000), and **collapsed the
diagnosis to one label**: `oxidation_damage` on all 200 dev cases from step 25, which is the
`cause_acc` of 0.140 -- exactly that label's share of the split. There is no signal in the cause for
a critic to track.

In [7]:
RUN = HERE / f"{PREFIX}_runs" / "ent0-s0"
ending = json.loads((RUN / "ending.json").read_text())
print(f"stopped at step {ending['stopped_at']}: {ending['reason']}\n")
print(f"  {'step':>4} {'reward':>7} {'flags':>6} {'cause':>6} {'schema':>6} {'critic - clock':>15}   causes said")
for r in map(json.loads, (RUN / "eval.jsonl").read_text().splitlines()):
    crit = "-" if r["critic_residual_median"] is None else f"{r['critic_residual_median']:+.4f}"
    print(f"  {r['step']:>4} {r['reward']:>7.4f} {r['flags_acc']:>6.3f} {r['cause_acc']:>6.3f} {r['schema_ok']:>6.3f} {crit:>15}   {r['predicted_cause_hist']}")
rows = [json.loads(l) for l in (RUN / "metrics.jsonl").read_text().splitlines()]
print("\n  critic - clock, median by window of training steps")
for a, b in ((15, 40), (40, 75)):
    res = [r["value_ev"] - r["value_ev_position"] for r in rows[a:b]
           if r["value_ev"] is not None and r["value_ev_position"] is not None]
    print(f"    steps {a}-{b - 1}: {st.median(res):+.4f}, ahead on {sum(x > 0 for x in res)}/{len(res)}")

stopped at step 75: critic gate: value_ev not above the position clock at two evaluations

  step  reward  flags  cause schema  critic - clock   causes said
     0  0.2200  0.360  0.150  0.925               -   {'scaling': 15, 'compaction': 182, 'biofouling': 3}
    25  0.2501  0.505  0.140  1.000         -0.0204   {'oxidation_damage': 200}
    50  0.2509  0.507  0.140  1.000         -0.0179   {'oxidation_damage': 200}
    75  0.2503  0.512  0.140  1.000         -0.0180   {'oxidation_damage': 200}

  critic - clock, median by window of training steps
    steps 15-39: -0.0003, ahead on 12/25
    steps 40-74: -0.0163, ahead on 6/35


## What the successful critics had

The same statistic -- critic minus clock, median by window -- read off the 1.7B series' own
`metrics.jsonl`, for every run whose critic worked. Over steps 15-39 they sat at the clock, between
-0.012 and +0.003 (13b ahead on 3 of 25 steps); **all of them pulled ahead between steps 40 and 74**
and stayed ahead.
The raw 0.8B run fell behind in exactly that window.

What they had in common is upstream of the critic: **the numbers were right before RL started**.
`09` had the harness compute the arithmetic (numeric 1.000 on the raw 1.7B); `12`-`14` had a
calculator and a seed that learned to call it (0.997). With the numbers right, whether a flag is
right is readable from the state, and a correct set of flags determines the cause -- so as the
policy got better at them there was something for the critic to read. The raw 0.8B has numeric 0.01.

In [8]:
OLD = HERE.parent / "smoke_test" / "runs"
PRECEDENT = [("09 flat x3 (v3: harness arithmetic)", "ppo-qwen3-17b-v3-flat3-s0"),
             ("12 calculator + seed", "ppo-on-tool-seed-warm-200"),
             ("13b", "ppo-tool-qwen-agent-scoped-200"),
             ("13b, second seed", "ppo-tool-qwen-agent-scoped-200-s1"),
             ("14 gamma 0.999", "ppo-tool-joint-seed-200-g0999")]

def window(rows, a, b):
    res = [r["value_ev"] - r["value_ev_position"] for r in rows if a <= r["step"] < b
           and r.get("value_ev") is not None and r.get("value_ev_position") is not None]
    return f"{st.median(res):+.4f} {sum(x > 0 for x in res):>2}/{len(res):<2}" if res else "      -     "

print(f"  critic - clock, median (steps ahead)  {'15-39':>13} {'40-74':>13} {'75-124':>13} {'125-199':>13}")
for label, run in PRECEDENT:
    rows = [json.loads(l) for l in (OLD / run / "metrics.jsonl").read_text().splitlines()]
    print(f"  {label:36} " + " ".join(f"{window(rows, a, b):>13}" for a, b in ((15, 40), (40, 75), (75, 125), (125, 200))))
rows = [json.loads(l) for l in (RUN / "metrics.jsonl").read_text().splitlines()]
print(f"  {'this notebook: raw 0.8B, ent0-s0':36} " + " ".join(f"{window(rows, a, b):>13}" for a, b in ((15, 40), (40, 75))))

  critic - clock, median (steps ahead)          15-39         40-74        75-124       125-199
  09 flat x3 (v3: harness arithmetic)  +0.0025 14/25 +0.0163 33/35 +0.0407 49/50 +0.0314 69/75
  12 calculator + seed                 -0.0087  6/25 +0.0059 32/35 +0.0131 49/50 +0.0079 47/75
  13b                                  -0.0111  3/25 +0.0034 26/35 +0.0088 47/50 +0.0225 73/75
  13b, second seed                     -0.0123  2/25 +0.0058 30/35 +0.0108 45/50 +0.0187 72/75
  14 gamma 0.999                       -0.0064  6/25 +0.0066 23/35 +0.0156 45/50 +0.0250 70/75
  this notebook: raw 0.8B, ent0-s0     -0.0003 12/25 -0.0163  6/35


## What this leaves for 02

The brief asked for a critic that works on the raw model with the prompt unchanged, and on this
model that critic does not exist: not for want of a layer, a learning rate or a window, but because
the return depends on the answer key and the raw model's state does not carry it. The 1.7B series
reached a working critic only after the arithmetic was right, and so does the next notebook: the
prompt stays v2's, the model gets a **calculator** and a **seed that learns only to call it**, and
the critic is tuned again on that seed -- where the same ridge finds, for the first time, something
beyond position to read.